# 07 – Model A: demand growth (LightGBM)

**Goal:** forecast next-year electricity demand growth per buurt.
**Input:** `Data/processed/model_a_table.csv` (rows with ≥ 50 connections)

## Setup
- Time split: train t = 2019–2021, validation t = 2022, test 1 = 2023→24 (old series), test 2 = 2025→26 (new series).
- Features only if known at t **and** available at forecast time → `smart_meter_pct`, `consume_only_pct`, `low_tariff_pct` excluded (missing from 2025); `n_rows` excluded (artefact of Liander's postcode grouping).
- LightGBM, objective L1 (= MAE), early stopping on validation. Metric: MAE of log growth (~ percentage points); ranking = Spearman.
- Note: the validation year 2022 was an energy-crisis year, not a typical year → early stopping may be less representative.

## Results
| | Test 1 (2023→24) | Test 2 (2025→26) |
|---|---|---|
| Best baseline | 6.53% (no change) | 4.56% (typical growth) |
| LightGBM (absolute growth) | 7.00%, ranking 0.05 | 4.34%, ranking 0.25 |

- **National swing dominates:** median growth +0.4% to +1.1% (2019–21), **−3.5% (2022), −3.2% (2023)** (energy crisis), **+2.9% (2025)**. The model trained on calm years missed the swings (bias up to +4%).
- **Local part only** (growth minus the year median), model trained on this relative target:
  - test 1: 0.0544 vs 0.0552 for "follow the country" (−1.4%), ranking 0.10
  - test 2: 0.0389 vs 0.0400 (−2.8%), ranking 0.22
  - early stopping after 40 trees → little learnable signal.

## Conclusion
- One-year-ahead local demand growth is dominated by the national trend and noise (SJV is an estimate; postcode grouping shifts). Local signal is real but weak.
- **Decision:** Model A = national trend scenario (low / central / high) + small local adjustment from the relative model, with uncertainty ranges (see notebook 09). Main analytical weight on Model B (stress).
- Limitation: CBS values back-filled into early years (notebook 04) leak some later information into the backtest.
- Future work: multi-year horizon, where slow drivers like electrification should matter more.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error

PROCESSED = Path("../Data/processed")
a = pd.read_csv(PROCESSED / "model_a_table.csv")

# features: known at year t AND available at forecast time (2026)
not_features = ["buurtcode", "year", "series", "target_log_growth", "target_clipped", "train_ok",
                "smart_meter_pct", "consume_only_pct", "low_tariff_pct"]
features = [c for c in a.columns if c not in not_features]
print(f"{len(features)} features:")
print(features)

# only rows usable for training/testing
d = a[a["target_log_growth"].notna() & a["train_ok"]].copy()
train = d[d["year"] <= 2021]
valid = d[d["year"] == 2022]
test1 = d[d["year"] == 2023]
test2 = d[d["year"] == 2025]
print(f"\nTrain {len(train):,} | Valid {len(valid):,} | Test1 (2023->24) {len(test1):,} | Test2 (2025->26) {len(test2):,}")

# baselines (evaluated on the real, unclipped target)
def report(name, part, pred):
    mae = mean_absolute_error(part["target_log_growth"], pred)
    print(f"  {name:32s} MAE = {mae:.4f}  (~{100 * mae:.2f}% error)")

for name, part in [("Test1 2023->24", test1), ("Test2 2025->26", test2)]:
    print(f"\n{name}:")
    report("Baseline A: no change (0)", part, np.zeros(len(part)))
    report("Baseline B: train median growth", part, np.full(len(part), train["target_clipped"].median()))
    report("Baseline C: same as last year", part, part["lag_log_growth"].fillna(0).clip(-0.5, 0.5))

50 features:
['n_rows', 'connections', 'total_kwh', 'kwh_per_conn', 'share_3phase_pct', 'share_cross_border_pct', 'lag_log_growth', 'lag_conn_growth', 'temp_mean_c_t', 'hdd_year_t', 'cdd_year_t', 'frost_days_t', 'solar_kwh_m2_year_t', 'temp_mean_c_tm1', 'hdd_year_tm1', 'cdd_year_tm1', 'frost_days_tm1', 'solar_kwh_m2_year_tm1', 'hdd_change', 'a_inw', 'a_hh', 'g_hhgro', 'bev_dich', 'a_woning', 'a_bedv', 'a_opp_ha', 'a_lan_ha', 'ste_mvs', 'ste_oad', 'p_1gezw', 'p_mgezw', 'p_koopw', 'p_huurw', 'p_leegsw', 'p_bjj2k', 'p_bjo2k', 'g_ele', 'g_gas', 'g_ink_pi', 'a_pau', 'g_pau_hh', 'p_bj_me10', 'p_bj_mi10', 'p_won_z', 'p_won_m', 'p_won_zs', 'p_won_ev', 'woz', 'cbs_code_found', 'g_ele_tr']

Train 15,285 | Valid 5,129 | Test1 (2023->24) 5,145 | Test2 (2025->26) 5,056

Test1 2023->24:
  Baseline A: no change (0)        MAE = 0.0653  (~6.53% error)
  Baseline B: train median growth  MAE = 0.0694  (~6.94% error)
  Baseline C: same as last year    MAE = 0.0672  (~6.72% error)

Test2 2025->26:
  Basel

In [2]:
import lightgbm as lgb
from scipy.stats import spearmanr

features = [f for f in features if f != "n_rows"]

model_a = lgb.LGBMRegressor(
    objective="l1",            # optimise the absolute error directly (robust to outliers)
    n_estimators=3000,
    learning_rate=0.03,
    num_leaves=31,
    min_child_samples=50,
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
model_a.fit(
    train[features], train["target_clipped"],
    eval_set=[(valid[features], valid["target_clipped"])],
    eval_metric="l1",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)],
)
print("Best number of trees:", model_a.best_iteration_)

for name, part in [("Test1 2023->24", test1), ("Test2 2025->26", test2)]:
    pred = model_a.predict(part[features])
    print(f"\n{name}:")
    report("LightGBM", part, pred)
    report("Best baseline (no change)", part, np.zeros(len(part)))
    rho = spearmanr(part["target_log_growth"], pred).correlation
    print(f"  Ranking check (Spearman): {rho:.3f}")

Training until validation scores don't improve for 100 rounds
[200]	valid_0's l1: 0.0512535
[400]	valid_0's l1: 0.0509608
Early stopping, best iteration is:
[329]	valid_0's l1: 0.0508609
Best number of trees: 329

Test1 2023->24:
  LightGBM                         MAE = 0.0700  (~7.00% error)
  Best baseline (no change)        MAE = 0.0653  (~6.53% error)
  Ranking check (Spearman): 0.054

Test2 2025->26:
  LightGBM                         MAE = 0.0434  (~4.34% error)
  Best baseline (no change)        MAE = 0.0497  (~4.97% error)
  Ranking check (Spearman): 0.246


In [3]:
d["pred"] = model_a.predict(d[features])

check = d.groupby("year").agg(
    rows=("buurtcode", "size"),
    actual_median=("target_log_growth", "median"),
    predicted_median=("pred", "median"),
).round(4)
check["bias"] = check["predicted_median"] - check["actual_median"]
print("Median growth per year (log; 0.01 is about 1%):")
print(check.to_string())

# how much of the error is the national swing, and how much is local?
for name, yr in [("Test1 2023->24", 2023), ("Test2 2025->26", 2025)]:
    p = d[d["year"] == yr]
    actual_rel = p["target_log_growth"] - p["target_log_growth"].median()
    pred_rel = p["pred"] - p["pred"].median()
    print(f"\n{name}:")
    print(f"  Total MAE:                        {mean_absolute_error(p['target_log_growth'], p['pred']):.4f}")
    print(f"  MAE after removing the year swing: {mean_absolute_error(actual_rel, pred_rel):.4f}")
    print(f"  'No change' after removing swing:  {mean_absolute_error(actual_rel, np.zeros(len(p))):.4f}")

Median growth per year (log; 0.01 is about 1%):
      rows  actual_median  predicted_median    bias
year                                               
2019  5083         0.0038            0.0064  0.0026
2020  5094         0.0062            0.0049 -0.0013
2021  5108         0.0109            0.0108 -0.0001
2022  5129        -0.0347            0.0001  0.0348
2023  5145        -0.0322            0.0073  0.0395
2025  5056         0.0289            0.0101 -0.0188

Test1 2023->24:
  Total MAE:                        0.0700
  MAE after removing the year swing: 0.0555
  'No change' after removing swing:  0.0552

Test2 2025->26:
  Total MAE:                        0.0434
  MAE after removing the year swing: 0.0394
  'No change' after removing swing:  0.0400


In [4]:
# target = growth minus that year's national median (the local part only)
year_median = d.groupby("year")["target_log_growth"].transform("median")
d["target_rel"] = (d["target_log_growth"] - year_median).clip(-0.5, 0.5)

train_r, valid_r = d[d["year"] <= 2021], d[d["year"] == 2022]

model_rel = lgb.LGBMRegressor(
    objective="l1", n_estimators=3000, learning_rate=0.03, num_leaves=31,
    min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    random_state=42, verbose=-1,
)
model_rel.fit(
    train_r[features], train_r["target_rel"],
    eval_set=[(valid_r[features], valid_r["target_rel"])],
    eval_metric="l1",
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)],
)
print("Best number of trees:", model_rel.best_iteration_)

for name, yr in [("Test1 2023->24", 2023), ("Test2 2025->26", 2025)]:
    p = d[d["year"] == yr]
    actual_rel = p["target_log_growth"] - p["target_log_growth"].median()
    pred_rel = model_rel.predict(p[features])
    print(f"\n{name}:")
    print(f"  Relative model MAE:            {mean_absolute_error(actual_rel, pred_rel):.4f}")
    print(f"  'Follow the country' MAE:      {mean_absolute_error(actual_rel, np.zeros(len(p))):.4f}")
    print(f"  Ranking check (Spearman):      {spearmanr(actual_rel, pred_rel).correlation:.3f}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[40]	valid_0's l1: 0.037863
Best number of trees: 40

Test1 2023->24:
  Relative model MAE:            0.0544
  'Follow the country' MAE:      0.0552
  Ranking check (Spearman):      0.098

Test2 2025->26:
  Relative model MAE:            0.0389
  'Follow the country' MAE:      0.0400
  Ranking check (Spearman):      0.216
